In [ ]:
'''
Cambiamos el modelo teórico de alfa, en vez de ser cte le metemos
una variación con el tiempo para que sea coherente con los residuos vistos
'''

import numpy as np
import pandas as pd
from scipy.optimize import minimize
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error


###
# Semilla
###
np.random.seed(3435) 

###
# Datos
###
try:
    df = pd.read_csv('pde_data.csv')
except FileNotFoundError:
    print("Error: Archivo 'pde_data.csv' no encontrado.")
    exit()

#Datos físicos del problema
R_int, R_ext = 1.50, 1.80
lam, h_ext = 3.0, 15.0
T_ini, T_amb, T_ext_0, theta = 1650.0, 35.0, 350.0, 0.00053

t_nodes = np.sort(df['tiempo_min'].unique())
r_nodes = np.sort(df['radio_m'].unique())
Nt, Nr = len(t_nodes), len(r_nodes)
dr = r_nodes[1] - r_nodes[0]

df_train, df_test = train_test_split(df, test_size=0.20, random_state=34)

###
# Funciones
###

def T_acero(t_min):
    '''
    Temperatura del acero dentro de la cuchara, y por ende, en las paredes
    interiores.

    Inputs:

        -t_min: Tiempo en minutos

    Returns:

        -Temperatura del acero (ºC)
    '''
    return T_amb + (T_ini - T_amb) * np.exp(-theta * t_min)

def resolver_edp_euler_implicito(params_escalados):
    '''
    Resolutor de Euler Implícito para difusividad constante.

    Inputs:

        -params: alfa a tomar para la resolución

    Outputs:

        T_sim: Array con las temperaturas simuladas
    '''
    # 1. Desescalamos para recuperar la física real
    a0 = params_escalados[0] * 1e-6
    a1 = params_escalados[1] * 1e-9
    a2 = params_escalados[2] * 1e-8  # Amplitud del coseno
    a3 = params_escalados[3] * 1e-3  # Frecuencia dependiente del tiempo

    T_sim = np.zeros((Nt, Nr))
    T_sim[0, :] = T_ini + ((r_nodes - R_int) / (R_ext - R_int)) * (T_ext_0 - T_ini)

    r_inc = r_nodes[1:] 

    for n in range(Nt - 1):
        dt = t_nodes[n+1] - t_nodes[n]
        T_prev = T_sim[n, 1:]
        T_dir_next = T_acero(t_nodes[n+1])
        t_actual = t_nodes[n] # Extraemos el tiempo actual
        
        # 2. Alfa con término trigonométrico dependiente del TIEMPO
        alpha_array = (a0 + a1 * T_prev + a2 * np.cos(a3 * t_actual)) * 60.0 
        
        # 3. Arrays locales de coeficientes
        A = (alpha_array * dt) / (dr**2)
        B = (alpha_array * dt) / (2 * r_inc * dr)
        
        d_main = (1 + 2 * A)
        d_up = -(A + B)[:-1]
        d_down = -(A - B)[1:]
        
        d_main[-1] += 2 * A[-1] * (dr * h_ext / lam)

        M = np.diag(d_main) + np.diag(d_up, 1) + np.diag(d_down, -1)
        M[-1, -2] = -2 * A[-1] 
        
        rhs = T_prev.copy()
        rhs[0] += (A[0] - B[0]) * T_dir_next
        rhs[-1] += 2 * A[-1] * (dr * h_ext / lam) * T_amb
        
        T_sim[n+1, 1:] = np.linalg.solve(M, rhs)
        T_sim[n+1, 0] = T_dir_next
        
    return T_sim

def funcion_coste(params_escalados):
    '''
    Función de coste adaptada para recibir un escalar.
    Se castiga que el alfa no sea coherente, y en otro caso 
    se evalúa por su error cuadrático medio.

    Inputs:

        -a0_s: Valor de alfa

    Outputs:

        -Error cuadrático medio
    '''
    a0 = params_escalados[0] * 1e-6
    a1 = params_escalados[1] * 1e-9
    a2 = params_escalados[2] * 1e-8
    
    # Castigo riguroso evaluando el peor caso posible del coseno (-1)
    T_test = np.linspace(T_amb, T_ini, 50)
    
    # Restamos el valor absoluto de la amplitud para asegurar que el mínimo absoluto siempre sea > 0
    alpha_test_min = a0 + a1 * T_test - abs(a2)
    
    if np.any(alpha_test_min <= 0): 
        return 1e9 
    
    T_sim = resolver_edp_euler_implicito(params_escalados)
    
    df_sim = pd.DataFrame({
        'tiempo_min': np.repeat(t_nodes, Nr),
        'radio_m': np.tile(r_nodes, Nt),
        'T_base': T_sim.flatten()
    })
    m = pd.merge(df_train, df_sim, on=['tiempo_min', 'radio_m'])
    return np.sqrt(mean_squared_error(m['T'], m['T_base']))

print("\n[Fase 1] Calibrando alpha (f(T, t)) con scipy.optimize.minimize...")

# Guess inicial escalado
guess_inicial = [5.0, 0.0, 0.0, 1.0]

# Límites escalados
limites = ((0.1, 100.0), (-100.0, 100.0), (-50.0, 50.0), (-10.0, 10.0))

res = minimize(funcion_coste, guess_inicial, method='L-BFGS-B', bounds=limites)

a0_opt = res.x[0] * 1e-6
a1_opt = res.x[1] * 1e-9
a2_opt = res.x[2] * 1e-8
a3_opt = res.x[3] * 1e-3

print(f"Alphas optimizados:")
print(f"A = {a0_opt:.4e}")
print(f"B = {a1_opt:.4e}")
print(f"C (Amplitud) = {a2_opt:.4e}")
print(f"D (Frecuencia en tiempo) = {a3_opt:.4e}")

# Fase 2: XGBoost
T_fis = resolver_edp_euler_implicito(res.x)

df_fis = pd.DataFrame({
    'tiempo_min': np.repeat(t_nodes, Nr),
    'radio_m': np.tile(r_nodes, Nt),
    'T_base': T_fis.flatten()
})

ds_train = pd.merge(df_train, df_fis, on=['tiempo_min', 'radio_m'])
ds_test = pd.merge(df_test, df_fis, on=['tiempo_min', 'radio_m'])

xgb_reg = xgb.XGBRegressor(n_estimators=1600, learning_rate=0.05, max_depth=6, subsample=0.7, random_state=42)
xgb_reg.fit(ds_train[['tiempo_min', 'radio_m']], ds_train['T'] - ds_train['T_base'])

ds_test['T_hibrida'] = ds_test['T_base'] + xgb_reg.predict(ds_test[['tiempo_min', 'radio_m']])
rmse_f = np.sqrt(mean_squared_error(ds_test['T'], ds_test['T_base']))
rmse_h = np.sqrt(mean_squared_error(ds_test['T'], ds_test['T_hibrida']))

print("\n" + "="*50)
print(f"RMSE Física (Euler Implícito - Alfa f(T, t)): {rmse_f:.4f} ºC")
print(f"RMSE Híbrido:                                   {rmse_h:.4f} ºC")
print("="*50)

###
# Dataset
###

print("\n[Fase 3] Generando predicciones completas...")

df_completo = df.copy()
df_completo['orden_original'] = df_completo.index
ds_completo = pd.merge(df_completo, df_fis, on=['tiempo_min', 'radio_m'], how='left')
ds_completo = ds_completo.sort_values('orden_original').drop(columns=['orden_original'])

residuos_totales = xgb_reg.predict(ds_completo[['tiempo_min', 'radio_m']])

ds_completo['T'] = ds_completo['T_base'] + residuos_totales

df_final = ds_completo[['tiempo_min', 'radio_m', 'T']]
df_final.to_csv('predicciones_completas.csv', index=False)
print("Archivo generado con éxito.")